# Import modules

In [ ]:
import pandas as pd

# Data preparation

In [ ]:
sample_annot = pd.read_csv('MasterMapping_MetImmune_03_16_2022_release.csv')
sample_annot

In [ ]:
sample_annot.RNAFile.value_counts()

In [ ]:
microarrays = ["GSE76297.hugene20st.gene_symbol.csv",
               "Cornell_PROSTATE.BatchAdj.hugene20st.gene_symbol.csv",
               "GSE37751.hugene10st.gene_symbol.csv",
               "GSE89076.Agilent8x60K.log2_transformed.gene_symbol.csv",
               "Multiregions.2Batchs.Sample.hg19KnownGene.tpm.gene_symbol.csv",
               "GSE26193.hgu133plus2.gene_symbol.csv",
               "GSE62452.hugene10st.gene_symbol.csv"

]

In [ ]:
sample_annot_microarrays = sample_annot[sample_annot['RNAFile'].isin(microarrays)]
sample_annot_microarrays = sample_annot_microarrays.drop_duplicates(subset=['RNAFile'], keep = 'first')
sample_annot_microarrays

In [ ]:
import os


#Transcriptomics
directory_path = 'pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/transcriptomics_processed'
file_names = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]

for file_i in file_names:
  print(file_i)

In [ ]:
#Metabolomics
directory_path = 'pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/metabolomics_processed'
file_names = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]

for file_i in file_names:
  print(file_i)

In [ ]:
samples_dict = sample_annot_microarrays[["MetabFile", "RNAFile"]].set_index("RNAFile").to_dict()["MetabFile"]
samples_dict

In [ ]:
met_genes = pd.read_csv('human_gem_associated_genes.csv')
met_genes

In [ ]:
import joblib

model = joblib.load('ElasticNet.pkl')
model_features = model.feature_names_in_
model_features

In [ ]:
import pickle

with open('y_train_features.pkl', 'rb') as file:
    y_features = pickle.load(file)

y_features = list(y_features)

In [ ]:
def merge_data(dictionary):

  for sample_rna, sample_met in dictionary.items():

    #Gene expression
    rna = pd.read_csv(f'pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/transcriptomics_processed/{sample_rna}', index_col = 0)
    rna.index.name = None
    rna = rna.T
    met_genes_select = met_genes[met_genes['Symbol'].isin(rna.columns)]
    met_genes_select = met_genes_select.Symbol.to_list()
    rna = rna[met_genes_select]
    rna = rna.T
    rna = pd.concat([met_genes.set_index('Symbol'), rna], axis = 'columns')
    rna = rna.drop(columns = 'gene_id')
    rna.index.name = None
    rna = rna.T
    rna = rna[model_features]

    met_id = sample_annot[sample_annot['RNAFile'].isin(microarrays)]
    met_id = met_id[['RNAID', 'MetabID']]
    met_id = met_id.set_index('RNAID')
    met_id.index.name = None
    rna = pd.concat([met_id, rna], axis = 'columns')
    rna = rna.set_index('MetabID')

    file_name = sample_rna.split('.')[0]
    rna.to_csv(f'{file_name}_gene_expression.csv')
    print(str(file_name)+ '_gene_expression.csv' + ' saved!')

    #Metabolomics
    cohort_metabolomics = samples_dict[sample_rna]
    metabolomics = pd.read_excel(f'{cohort_metabolomics}', index_col = 0)
    common_metabolites = list(set(y_features).intersection(metabolomics.index))
    metabolomics = metabolomics.loc[common_metabolites]
    metabolomics = metabolomics.T

    metabolomics.to_csv(f'{file_name}_metabolomics.csv')
    print(str(file_name)+ '_metabolomics.csv' + ' saved!')

In [ ]:
merge_data(samples_dict)